## 04. Model Evaluation

In [26]:
from pathlib import Path
import time
import sys

sys.path.append(str(Path.cwd().parents[0]))
from src.validation import top_words_per_class_rf, top_words_per_class_nb

In [27]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.base import BaseEstimator
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/sms_data_cleaned.csv')

X = df['text_transformed']
y = df['target']

vectroizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(sgkf.split(X, y, groups=df['dedup_group_id']))
train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

y_train = train_df['target']
y_test = test_df['target']

X_train = train_df['text_transformed']
X_test = test_df['text_transformed']

In [28]:
# parameters: {'clf__alpha': 0.5, 'clf__fit_prior': False}

nb = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)),
    ('nb', MultinomialNB(alpha=0.5, fit_prior=False))
])

nb_start_time = time.perf_counter()
nb.fit(X_train, y_train)
nb_end_time= time.perf_counter()
nb_predict = nb.predict(X_test)
nb_training_time = nb_end_time - nb_start_time
nb_proba = nb.predict_proba(X_test)

# parameters: {'clf__class_weight': None, 'clf__max_depth': 60, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 5, 'clf__n_estimators': 200}

rfc = Pipeline([
    ('vectorizer', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)),
    ('nb', RandomForestClassifier(
    class_weight= None,
    max_depth= 60,
    max_features='sqrt',
    min_samples_leaf=1,
    min_samples_split=5,
    n_estimators=200))
])
rfc_start = time.perf_counter()
rfc.fit(X_train, y_train)
rfc_end = time.perf_counter()
rfc_training = rfc_end - rfc_start
rfc_predict = rfc.predict(X_test)
rfc_prob = rfc.predict_proba(X_test)

print("Model Training Time:")
print("====================")
print(f"Naye Bayes: {nb_training_time}")
print(f"Random Forest: {rfc_training}")

Model Training Time:
Naye Bayes: 0.026189099997282028
Random Forest: 0.6463360999478027


In [29]:
# labels: 0 - Legit, 1 - Scam, 2 - Spam
def label_predict_check(model: BaseEstimator, df: pd.DataFrame, label: int, sample_n: int | None = 10):

    if label in [0, 1, 2]:
        labeled_df = df[df['target'] == label]
    else:
        raise ValueError("The target label is not within the target column")
    # determines whether a specific text from a label is predicted as also the actuals

    if sample_n is not None:
        n_sample = min(sample_n, len(labeled_df))
        sampled = labeled_df.sample(n_sample, replace=False, random_state=42)
    else:
        sampled = labeled_df

    # values:
    sampled_txts = sampled['text_transformed'].values
    predictions = model.predict(sampled_txts)
    actuals = sampled['target'].values

    result_df = pd.DataFrame({
        "Texts" : sampled_txts, "Actual" : actuals, "Predicted" : predictions
    })

    result_df['Flag'] = np.where(result_df['Predicted'] == result_df['Actual'], "Matched", "Not Matched")

    return result_df

## Naive Bayes Feature Significance per Target

In [30]:
target_names = ['Legitimate', 'Scam', 'Spam']

In [31]:
top_words_per_class_nb(nb, target_names, 20)

--- Top 20 words for Legitimate (NB log-odds) ---
content                    log-odds=2.978
content supported          log-odds=2.978
supported                  log-odds=2.978
tutor                      log-odds=1.894
youtube                    log-odds=1.894
download app               log-odds=1.647
delivery                   log-odds=1.564
foodpanda                  log-odds=1.520
download sign              log-odds=1.496
araw makatanggap           log-odds=1.489
post                       log-odds=1.486
recharge moneytoken        log-odds=1.485
makakakuha                 log-odds=1.477
habang nanonood            log-odds=1.476
bawat araw                 log-odds=1.476
nanonood                   log-odds=1.476
moneytoken bawat           log-odds=1.476
nanonood youtube           log-odds=1.476
kumita habang              log-odds=1.476
araw makipag               log-odds=1.476

--- Top 20 words for Scam (NB log-odds) ---
points expire              log-odds=2.105
expire today           

## Random Forests Feature Significance per Target

In [32]:
target_names = ['Legitimate', 'Scam', 'Spam']
top_words_per_class_rf(rfc, X_test[:500], y_test[:500], target_names, 10)

Top words for each class in Random Forests:

--- Top 10 words for Legitimate ---
bonus                    : 0.1833 +/- 0.0247
urltoken                 : 0.0165 +/- 0.0107
win                      : 0.0130 +/- 0.0221
contact                  : 0.0123 +/- 0.0026
account                  : 0.0120 +/- 0.0126
office                   : 0.0110 +/- 0.0000
email                    : 0.0110 +/- 0.0000
login                    : 0.0086 +/- 0.0044
casino                   : 0.0086 +/- 0.0044
get                      : 0.0082 +/- 0.0078

--- Top 10 words for Scam ---
bonus                    : 0.1527 +/- 0.0185
win                      : 0.0192 +/- 0.0085
new                      : 0.0123 +/- 0.0034
get                      : 0.0116 +/- 0.0025
account                  : 0.0115 +/- 0.0038
bet                      : 0.0111 +/- 0.0095
spin                     : 0.0110 +/- 0.0027
gcash                    : 0.0088 +/- 0.0024
money                    : 0.0088 +/- 0.0038
real name                : 0.0077

## Classification Metrics

In [44]:
nb = pd.DataFrame({
    'Accuracy Score' : [accuracy_score(y_test, nb_predict)],
    'Precision Score' : [precision_score(y_test, nb_predict, average='macro')],
    'Recall Score' : [recall_score(y_test, nb_predict, average='macro')],
    'F1 Score' : [f1_score(y_test, nb_predict, average='macro')],
    'ROC AUC Score' : [roc_auc_score(y_test, nb_proba, multi_class='ovr')],
    'Training Time' : [nb_training_time]
})

rf = pd.DataFrame({
    'Accuracy Score' : [accuracy_score(y_test, rfc_predict)],
    'Precision Score' : [precision_score(y_test, rfc_predict, average='macro')],
    'Recall Score' : [recall_score(y_test, rfc_predict, average='macro')],
    'F1 Score' : [f1_score(y_test, rfc_predict, average='macro')],
    'ROC AUC Score' : [roc_auc_score(y_test, rfc_prob, multi_class='ovr')],
    'Training Time' : [rfc_training]
})

## McNemar's Statistical Test